In [ ]:
!pip install librosa numpy sounddevice scipy scikit-learn tensorflow pyaudio

In [ ]:
import os
import random
import librosa
import numpy as np
import sounddevice as sd
from scipy.io.wavfile import write
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import load_model
import sys
sys.path.append(os.path.abspath('..'))

In [7]:
#This code is used to check your device Imdex.
import pyaudio
p = pyaudio.PyAudio()
print(" AVAILABLE MICROPHONES:\n" + "="*30)

for i in range(p.get_device_count()):
    dev_info = p.get_device_info_by_index(i)
    
    # Check if the device has input channels (meaning it's a microphone)
    if dev_info.get('maxInputChannels') > 0:
        print(f" Device Index: {i}")
        print(f" Name: {dev_info.get('name')}")
        print(f" Max Channels: {int(dev_info.get('maxInputChannels'))}")
        print(f" Native Rate: {int(dev_info.get('defaultSampleRate'))} Hz")
        print("-" * 30)

p.terminate()

 AVAILABLE MICROPHONES:
 Device Index: 0
 Name: MacBook Air Microphone
 Max Channels: 1
 Native Rate: 44100 Hz
------------------------------


In [ ]:
import os
import time
import librosa
import numpy as np
import pyaudio
import joblib
from tensorflow.keras.models import load_model


SAMPLE_RATE = 22050
DURATION = 3
FRAME_LENGTH = 2048
HOP_LENGTH = 512


MODEL_PATH = 'nepali_model_artifacts/nepali_lstm_model.h5'
SCALER_PATH = 'nepali_model_artifacts/scaler.joblib'
ENCODER_PATH = 'nepali_model_artifacts/label_encoder.joblib'


print("Loading Model, Scaler, and Encoder...")
try:
    model = load_model(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    le = joblib.load(ENCODER_PATH)
    print(" All artifacts loaded successfully!\n")
except Exception as e:
    print(f" Error loading artifacts: {e}")
    print("Please ensure the artifacts folder is in the same directory as this script.")
    exit()


def process_live_audio(audio, sample_rate):
    audio, _ = librosa.effects.trim(audio, top_db=20)
    if len(audio) > 0:
        audio = librosa.util.normalize(audio)
        
    target_length = int(sample_rate * DURATION)
    if len(audio) < target_length:
        audio = np.pad(audio, (0, max(0, target_length - len(audio))), "constant")
    else:
        audio = audio[:target_length]
    
    zcr_val = librosa.feature.zero_crossing_rate(audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH).T
    rmse_val = librosa.feature.rms(y=audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH).T
    mfcc_val = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=13, hop_length=HOP_LENGTH).T
    
    sequence = np.hstack((zcr_val, rmse_val, mfcc_val))
    return sequence

def record_audio_mac(duration=DURATION, target_sr=SAMPLE_RATE):
    p = pyaudio.PyAudio()
    
    RATE = 44100 
    CHANNELS = 1 
    DEVICE_INDEX = 0 # <------------------- Enter your device_index here
    
    CHUNK = 1024
    FORMAT = pyaudio.paFloat32

    print(f" RECORDING FOR {duration} SECONDS... ")
    
    stream = p.open(format=FORMAT, 
                    channels=CHANNELS, 
                    rate=RATE, 
                    input=True, 
                    input_device_index=DEVICE_INDEX,
                    frames_per_buffer=CHUNK)

    frames = []

    for i in range(0, int(RATE / CHUNK * duration)):
        data = stream.read(CHUNK, exception_on_overflow=False)
        frames.append(np.frombuffer(data, dtype=np.float32))

    print("Recording complete. Processing...\n")

    stream.stop_stream()
    stream.close()
    p.terminate()

    audio_data = np.hstack(frames)

    if RATE != target_sr:
        audio_data = librosa.resample(y=audio_data, orig_sr=RATE, target_sr=target_sr)

    return audio_data

def live_predict():
    print("Get ready to speak...")
    time.sleep(1)
    
    audio = record_audio_mac()
    
    features = process_live_audio(audio, SAMPLE_RATE)
    
    features_scaled = scaler.transform(features)
    
    lstm_input = np.expand_dims(features_scaled, axis=0)
    
    predictions = model.predict(lstm_input, verbose=0)
    predicted_index = np.argmax(predictions, axis=1)[0]
    confidence = np.max(predictions) * 100
    
    predicted_word = le.inverse_transform([predicted_index])[0]
    print("         LIVE PREDICTION RESULT         ")
    print(f"  Predicted:  {predicted_word}")
    print(f"  Confidence: {confidence:.2f}%")



Loading Model, Scaler, and Encoder...
 All artifacts loaded successfully!



/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
if __name__ == "__main__":
    while True:
        live_predict()
        
        user_input = input("Press ENTER to record again, or type 'q' to quit: ")
        if user_input.lower() == 'q':
            break

Get ready to speak...
 RECORDING FOR 4.224 SECONDS... 
Recording complete. Processing...

         LIVE PREDICTION RESULT         
  Predicted:  Tapaiko_Ghar_Kaha_Xa
  Confidence: 99.99%
Get ready to speak...
 RECORDING FOR 4.224 SECONDS... 
Recording complete. Processing...

         LIVE PREDICTION RESULT         
  Predicted:  Tapaiko_Ghar_Kaha_Xa
  Confidence: 51.86%
Get ready to speak...
 RECORDING FOR 4.224 SECONDS... 
Recording complete. Processing...

         LIVE PREDICTION RESULT         
  Predicted:  Yo_Hamro_AI_Ko_Project_ho
  Confidence: 97.39%
Get ready to speak...
 RECORDING FOR 4.224 SECONDS... 
Recording complete. Processing...

         LIVE PREDICTION RESULT         
  Predicted:  Ram_Le_Vaat_Khanxa
  Confidence: 96.69%
Get ready to speak...
 RECORDING FOR 4.224 SECONDS... 
Recording complete. Processing...

         LIVE PREDICTION RESULT         
  Predicted:  Yo_Hamro_AI_Ko_Project_ho
  Confidence: 63.35%
Get ready to speak...
 RECORDING FOR 4.224 SECONDS... 
Re